## Google Colab Setup

Mount Google Drive, pull data + utility script from GitHub, then train.
The setup cell also downloads a fresh fine-tuning ESM-2 snapshot from Hugging Face on every run so the notebook never reuses a stale local cache.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── Colab output directories on Drive ────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
DRIVE_MODELS = DRIVE_ROOT / 'models'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive directories ready:')
print(f'  {DRIVE_MODELS}')
print(f'  {DRIVE_RESULTS}')


Mounted at /content/drive
Drive directories ready:
  /content/drive/MyDrive/XAllergen2.0/models
  /content/drive/MyDrive/XAllergen2.0/results


In [2]:
import os
import urllib.error
import urllib.request
from pathlib import Path

from huggingface_hub import snapshot_download

RUNTIME_ROOT = Path('/content/XAllergen2.0')
DATA_DIR   = RUNTIME_ROOT / 'data'
MODEL_DIR  = RUNTIME_ROOT / 'models'
RESULTS_DIR = RUNTIME_ROOT / 'results'
FRESH_ESM2_DIR = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'
for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR, FRESH_ESM2_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'
UTILS_PATH = RUNTIME_ROOT / 'finetune_notebook_utils.py'

# Utility script
try:
    urllib.request.urlretrieve(f'{RAW}/finetune_notebook_utils.py', UTILS_PATH)
    print('Downloaded utility script from GitHub.')
except urllib.error.HTTPError as exc:
    if exc.code != 404:
        raise
    UTILS_PATH.write_text(
        'from __future__ import annotations\n\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom transformers import AutoTokenizer, EsmModel\n\n\nRANDOM_STATE = 42\nESM_MODEL_NAME = "esm2_t6_8M_UR50D"\nHF_MODEL_NAME = f"facebook/{ESM_MODEL_NAME}"\nHIDDEN_DIM = 128\nDROPOUT = 0.3\nTHRESHOLD = 0.5\nIG_STEPS = 50\nMAX_SEQ_LEN = 1022\n\n\ndef configure_matplotlib_cache(cwd: Path) -> None:\n    mplconfigdir = cwd.resolve() / ".matplotlib"\n    mplconfigdir.mkdir(exist_ok=True)\n    os.environ.setdefault("MPLCONFIGDIR", str(mplconfigdir))\n\n\ndef resolve_hf_model_source(model_name: str = HF_MODEL_NAME) -> str:\n    override = os.environ.get("XALLERGEN_HF_MODEL_DIR")\n    if override:\n        return override\n\n    repo_dir = Path.home() / ".cache" / "huggingface" / "hub" / f"models--{model_name.replace(\'/\', \'--\')}"\n    snapshots_dir = repo_dir / "snapshots"\n    if snapshots_dir.exists():\n        for snapshot in sorted(snapshots_dir.iterdir(), reverse=True):\n            if snapshot.is_dir() and any(\n                list(snapshot.glob(pat))\n                for pat in ("*.safetensors", "*.bin")\n            ):\n                return str(snapshot)\n\n    return model_name\n\n\ndef find_project_root(start: Path) -> Path:\n    for candidate in [start, *start.parents]:\n        if (candidate / "data").exists():\n            return candidate\n    raise FileNotFoundError(\n        "Could not locate project root. Make sure VSCode is opened from inside the XAllergen2.0 folder."\n    )\n\n\ndef seed_everything(seed: int = RANDOM_STATE) -> None:\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    random.seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef detect_device() -> str:\n    if torch.cuda.is_available():\n        return "cuda"\n    if torch.backends.mps.is_available():\n        return "mps"\n    return "cpu"\n\n\ndef print_runtime_context(device: str, project_root: Path) -> None:\n    run_target = os.environ.get("XALLERGEN_RUN_TARGET", "local")\n    print(f"RUN_TARGET: {run_target}")\n    print(f"Device: {device}")\n    print(f"Project root: {project_root}")\n    if device == "cuda":\n        print("GPU configuration:")\n        print(f"  backend: CUDA")\n        print(f"  CUDA available: {torch.cuda.is_available()}")\n        print(f"  GPU count: {torch.cuda.device_count()}")\n        if torch.cuda.is_available():\n            print(f"  Current device: {torch.cuda.get_device_name(torch.cuda.current_device())}")\n    elif device == "mps":\n        print("GPU configuration:")\n        print("  backend: Apple Metal Performance Shaders (MPS)")\n        print(f"  built with MPS: {torch.backends.mps.is_built()}")\n        print(f"  MPS available: {torch.backends.mps.is_available()}")\n    else:\n        print("GPU configuration:")\n        print("  No MPS accelerator detected. Running on CPU.")\n\n\ndef build_tokenizer(model_name: str = HF_MODEL_NAME):\n    return AutoTokenizer.from_pretrained(resolve_hf_model_source(model_name))\n\n\nclass AttentionPooling(nn.Module):\n    def __init__(self, embed_dim: int):\n        super().__init__()\n        self.score = nn.Linear(embed_dim, 1)\n\n    def forward(\n        self, residue_embeddings: torch.Tensor, attention_mask: torch.Tensor\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        mask = attention_mask.bool().expand(residue_embeddings.shape[0], -1)\n        scores = self.score(residue_embeddings).squeeze(-1)\n        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)\n        weights = torch.softmax(scores, dim=-1)\n        pooled = torch.sum(weights.unsqueeze(-1) * residue_embeddings, dim=1)\n        return pooled, weights\n\n\ndef _freeze_module(module: nn.Module) -> None:\n    for param in module.parameters():\n        param.requires_grad = False\n\n\ndef _unfreeze_module(module: nn.Module) -> None:\n    for param in module.parameters():\n        param.requires_grad = True\n\n\ndef _normalize_layer_indices(layer_indices: list[int] | tuple[int, ...] | None, num_layers: int) -> list[int]:\n    if layer_indices is None:\n        return []\n\n    normalized = []\n    for idx in layer_indices:\n        normalized_idx = idx if idx >= 0 else num_layers + idx\n        if normalized_idx < 0 or normalized_idx >= num_layers:\n            raise ValueError(\n                f"Layer index {idx} is out of range for an ESM-2 backbone with {num_layers} layers."\n            )\n        normalized.append(normalized_idx)\n    return sorted(set(normalized))\n\n\ndef configure_esm2_trainable_layers(\n    backbone: EsmModel,\n    unfreeze_backbone: bool = True,\n    trainable_layer_indices: list[int] | tuple[int, ...] | None = None,\n    unfreeze_embeddings: bool = False,\n    unfreeze_pooler: bool = False,\n) -> list[int]:\n    _freeze_module(backbone)\n\n    encoder_layers = backbone.encoder.layer\n    selected_layers = _normalize_layer_indices(trainable_layer_indices, len(encoder_layers))\n\n    if unfreeze_backbone:\n        _unfreeze_module(backbone)\n        return list(range(len(encoder_layers)))\n\n    if unfreeze_embeddings:\n        _unfreeze_module(backbone.embeddings)\n\n    for idx in selected_layers:\n        _unfreeze_module(encoder_layers[idx])\n\n    if unfreeze_pooler and getattr(backbone, "pooler", None) is not None:\n        _unfreeze_module(backbone.pooler)\n\n    return selected_layers\n\n\nclass FinetunedESMAllergenClassifier(nn.Module):\n    def __init__(\n        self,\n        model_name: str,\n        hidden_dim: int = HIDDEN_DIM,\n        dropout: float = DROPOUT,\n        unfreeze_backbone: bool = True,\n        trainable_layer_indices: list[int] | tuple[int, ...] | None = None,\n        unfreeze_embeddings: bool = False,\n        unfreeze_pooler: bool = False,\n    ):\n        super().__init__()\n        self.backbone = EsmModel.from_pretrained(resolve_hf_model_source(model_name))\n        self.trainable_layer_indices = configure_esm2_trainable_layers(\n            self.backbone,\n            unfreeze_backbone=unfreeze_backbone,\n            trainable_layer_indices=trainable_layer_indices,\n            unfreeze_embeddings=unfreeze_embeddings,\n            unfreeze_pooler=unfreeze_pooler,\n        )\n        embed_dim = self.backbone.config.hidden_size\n        self.attention_pool = AttentionPooling(embed_dim)\n        self.classifier = nn.Sequential(\n            nn.Linear(embed_dim, hidden_dim),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, 1),\n        )\n\n    def forward_from_residue_embeddings(\n        self, residue_embeddings: torch.Tensor, attention_mask: torch.Tensor\n    ) -> dict:\n        pooled, attention_weights = self.attention_pool(residue_embeddings, attention_mask)\n        logits = self.classifier(pooled).squeeze(-1)\n        return {\n            "logits": logits,\n            "attention_weights": attention_weights,\n            "residue_embeddings": residue_embeddings,\n        }\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> dict:\n        residue_embeddings = self.backbone(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n        ).last_hidden_state\n        return self.forward_from_residue_embeddings(residue_embeddings, attention_mask)\n\n    def forward_from_inputs_embeds(\n        self, inputs_embeds: torch.Tensor, attention_mask: torch.Tensor\n    ) -> dict:\n        token_dropout_backup = self.backbone.embeddings.token_dropout\n        self.backbone.embeddings.token_dropout = False\n        try:\n            residue_embeddings = self.backbone(\n                inputs_embeds=inputs_embeds,\n                attention_mask=attention_mask,\n            ).last_hidden_state\n        finally:\n            self.backbone.embeddings.token_dropout = token_dropout_backup\n        return self.forward_from_residue_embeddings(residue_embeddings, attention_mask)\n\n\ndef tokenize_sequence(tokenizer, sequence: str, device: str) -> dict:\n    encodings = tokenizer(\n        sequence,\n        add_special_tokens=False,\n        padding=False,\n        truncation=False,\n        return_tensors="pt",\n    )\n    return {\n        "input_ids": encodings["input_ids"].to(device),\n        "attention_mask": encodings["attention_mask"].to(device),\n    }\n\n\ndef load_finetune_checkpoint(\n    checkpoint_path: Path,\n    device: str,\n    model_name: str = HF_MODEL_NAME,\n    hidden_dim: int = HIDDEN_DIM,\n    dropout: float = DROPOUT,\n    unfreeze_backbone: bool = True,\n    trainable_layer_indices: list[int] | tuple[int, ...] | None = None,\n    unfreeze_embeddings: bool = False,\n    unfreeze_pooler: bool = False,\n) -> tuple[FinetunedESMAllergenClassifier, dict]:\n    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n    architecture = checkpoint.get(\n        "architecture_hyperparameters",\n        {"hidden_dim": hidden_dim, "dropout": dropout},\n    )\n    model = FinetunedESMAllergenClassifier(\n        model_name,\n        hidden_dim=architecture.get("hidden_dim", hidden_dim),\n        dropout=architecture.get("dropout", dropout),\n        unfreeze_backbone=unfreeze_backbone,\n        trainable_layer_indices=trainable_layer_indices,\n        unfreeze_embeddings=unfreeze_embeddings,\n        unfreeze_pooler=unfreeze_pooler,\n    ).to(device)\n    incompatible = model.load_state_dict(checkpoint["model_state_dict"], strict=False)\n    if incompatible.missing_keys:\n        non_positional = [k for k in incompatible.missing_keys if "position_embeddings" not in k]\n        if non_positional:\n            raise RuntimeError(\n                "Unexpected missing keys in checkpoint:\\n"\n                + "\\n".join(f"  {k}" for k in non_positional)\n            )\n    if incompatible.unexpected_keys:\n        raise RuntimeError(\n            "Unexpected extra keys in checkpoint:\\n"\n            + "\\n".join(f"  {k}" for k in incompatible.unexpected_keys)\n        )\n    model.eval()\n    return model, checkpoint\n\n\ndef normalize_scores(scores: np.ndarray) -> np.ndarray:\n    scores = np.asarray(scores, dtype=np.float64)\n    scores = np.maximum(scores, 0.0)\n    total = scores.sum()\n    return scores / total if total > 0 else scores\n\n\ndef compute_attention_weights(model, tokenizer, sequence: str, device: str) -> np.ndarray:\n    encodings = tokenize_sequence(tokenizer, sequence, device)\n    with torch.no_grad():\n        outputs = model(encodings["input_ids"], encodings["attention_mask"])\n    weights = outputs["attention_weights"].squeeze(0).detach().cpu().numpy()\n    valid_length = int(encodings["attention_mask"].sum().item())\n    return weights[:valid_length]\n\n\ndef compute_integrated_gradients(\n    model,\n    tokenizer,\n    sequence: str,\n    device: str,\n    steps: int = IG_STEPS,\n    normalize: bool = False,\n) -> np.ndarray:\n    from captum.attr import IntegratedGradients\n\n    encodings = tokenize_sequence(tokenizer, sequence, device)\n    attention_mask = encodings["attention_mask"]\n\n    input_embeds = model.backbone.get_input_embeddings()(encodings["input_ids"]).detach()\n    baseline = torch.zeros_like(input_embeds)\n\n    def ig_forward(inputs_embeds: torch.Tensor) -> torch.Tensor:\n        return model.forward_from_inputs_embeds(inputs_embeds, attention_mask)["logits"]\n\n    attributions = IntegratedGradients(ig_forward).attribute(\n        inputs=input_embeds,\n        baselines=baseline,\n        n_steps=steps,\n    )\n    importance = attributions.abs().sum(dim=-1).squeeze(0).detach().cpu().numpy()\n    valid_length = int(attention_mask.sum().item())\n    importance = importance[:valid_length]\n    return normalize_scores(importance) if normalize else importance\n\n\ndef mean_metric_dicts(metric_rows: list[dict]) -> dict:\n    return {\n        "auroc": float(np.nanmean([row["auroc"] for row in metric_rows])),\n        "auprc": float(np.mean([row["auprc"] for row in metric_rows])),\n        "precision_at_k": float(np.mean([row["precision_at_k"] for row in metric_rows])),\n    }\n',
        encoding='utf-8',
    )
    print('GitHub copy of finetune_notebook_utils.py was not found; wrote bundled fallback instead.')

# Always download a fresh fine-tuning ESM-2 snapshot and force downstream code to use it
FRESH_ESM2_PATH = snapshot_download(
    repo_id='facebook/esm2_t6_8M_UR50D',
    local_dir=FRESH_ESM2_DIR,
    local_dir_use_symlinks=False,
    force_download=True,
    resume_download=False,
)
os.environ['XALLERGEN_HF_MODEL_DIR'] = str(FRESH_ESM2_PATH)

# Data files
DATA_FILES = [
    'deepalgpro_test_cleaned.csv',
    'deepalgpro_train_cleaned.csv',
    'negatives_splitA.csv',
    'negatives_splitB.csv',
    'positives_splitA.csv',
    'positives_splitB.csv',
]
for fname in DATA_FILES:
    urllib.request.urlretrieve(f'{RAW}/data/{fname}', DATA_DIR / fname)

print('Downloaded:')
print(f'  Utility script: {UTILS_PATH}')
print(f'  Fresh ESM-2 snapshot: {FRESH_ESM2_PATH}')
for p in sorted(RUNTIME_ROOT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(RUNTIME_ROOT)}')


Downloaded utility script from GitHub.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded:
  Utility script: /content/XAllergen2.0/finetune_notebook_utils.py
  Fresh ESM-2 snapshot: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
  data/deepalgpro_test_cleaned.csv
  data/deepalgpro_train_cleaned.csv
  data/negatives_splitA.csv
  data/negatives_splitB.csv
  data/positives_splitA.csv
  data/positives_splitB.csv
  finetune_notebook_utils.py
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/.gitignore
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/CACHEDIR.TAG
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/.gitattributes.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/README.md.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/config.json.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/model.safetensors.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.cache/huggingface/download/pytorch_model.bin.metadata
  hf_models/facebook_esm2_t6_8M_UR50D/.ca

# Fine-tuned ESM-2 baseline for DeepAlgPro

This notebook trains a fine-tuned `esm2_t6_8M_UR50D` allergenicity classifier on `data/deepalgpro_train_cleaned.csv` and evaluates it on the held-out `data/deepalgpro_test_cleaned.csv`.

Architecture:
- Backbone: `esm2_t6_8M_UR50D`, unfrozen and fine-tuned throughout training.
- Attention pooling: `Linear(embed_dim -> 1)` per residue, softmax-normalized across sequence length, then a weighted sum of residue embeddings. The resulting per-residue weights are kept as an attribution signal.
- Classification head: `Linear(embed_dim -> 128) -> ReLU -> Dropout(0.3) -> Linear(128 -> 1)`.
- Training loss: `BCEWithLogitsLoss`; inference uses `sigmoid`.

Split strategy:
- `deepalgpro_train_cleaned.csv` is split into train and validation with a stratified random 90/10 split.
- `sklearn.model_selection.train_test_split` is used with `stratify=df["label"]` and `random_state=42`.
- `deepalgpro_test_cleaned.csv` remains fully held out until final evaluation.


In [ ]:
import json
import math
import sys
import time as _time
from pathlib import Path

# Allow imports from the src-layout package without a root-level shim.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break

# Add runtime root to path so finetune_notebook_utils is importable
RUNTIME_ROOT = Path('/content/XAllergen2.0')
if str(RUNTIME_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNTIME_ROOT))

from xallergen.finetune_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    RANDOM_STATE,
    THRESHOLD,
    FinetunedESMAllergenClassifier,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    load_finetune_checkpoint,
    normalize_scores,
    seed_everything,
)

import matplotlib
matplotlib.use('Agg')  # headless-safe on Colab
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ── Hyperparameters ───────────────────────────────────────────────────────
BATCH_SIZE    = 24
EPOCHS        = 15
PATIENCE      = 5
LEARNING_RATE = 4e-6
WEIGHT_DECAY  = 1e-4

# ── Runtime paths ─────────────────────────────────────────────────────────
DATA_DIR    = RUNTIME_ROOT / 'data'
MODEL_DIR   = RUNTIME_ROOT / 'models'
RESULTS_DIR = RUNTIME_ROOT / 'results'

TRAIN_CSV       = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV        = DATA_DIR / 'deepalgpro_test_cleaned.csv'
CHECKPOINT_PATH = MODEL_DIR / 'finetuned_esm2_8e-6.pt'
METRICS_PATH    = RESULTS_DIR / 'finetuned_esm2_8e-6_metrics.json'

# ── Drive output paths (set in cell 0) ───────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/XAllergen2.0')
DRIVE_MODELS  = DRIVE_ROOT / 'models'
DRIVE_RESULTS = DRIVE_ROOT / 'results'

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [4]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df["sequence_id"] = df["sequence_id"].astype(str)
    df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
    df["label"] = df["label"].astype(int)

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df["label"],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df = val_split_df.reset_index(drop=True).copy()

print(f"Train sequences: {len(train_split_df):,}")
print(train_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Validation sequences: {len(val_split_df):,}")
print(val_split_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())
print()
print(f"Held-out test sequences: {len(test_df):,}")
print(test_df['label'].value_counts().sort_index().rename(index={0: 'negative', 1: 'positive'}).to_string())

Train sequences: 5,011
label
negative    2505
positive    2506

Validation sequences: 557
label
negative    279
positive    278

Held-out test sequences: 1,376
label
negative    688
positive    688


In [8]:
class SequenceDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        return {
            "sequence_id": row["sequence_id"],
            "sequence": row["sequence"],
            "label": int(row["label"]),
        }


tokenizer = build_tokenizer(HF_MODEL_NAME)


def collate_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    return {
        "sequence_id": [item["sequence_id"] for item in batch],
        "sequence": sequences,
        "label": torch.tensor([item["label"] for item in batch], dtype=torch.float32),
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"]
    }


def move_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    moved["input_ids"] = batch["input_ids"].to(device)
    moved["attention_mask"] = batch["attention_mask"].to(device)
    moved["label"] = batch["label"].to(device)
    return moved


train_loader = DataLoader(SequenceDataset(train_split_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_batch)
val_loader = DataLoader(SequenceDataset(val_split_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(SequenceDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_batch)

model = FinetunedESMAllergenClassifier(
    HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    unfreeze_backbone=False,
    trainable_layer_indices=[-1],
    unfreeze_embeddings=False,
    unfreeze_pooler=False,
    dropout=DROPOUT
).to(DEVICE)

assert any(param.requires_grad for param in model.backbone.parameters())
print("Trainable ESM layers:", model.trainable_layer_indices)

trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Backbone hidden size: {model.backbone.config.hidden_size}")


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable ESM layers: [5]
Trainable parameter tensors: 22
Backbone hidden size: 320


In [9]:
from typing import Optional, Tuple
from tqdm.auto import tqdm

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
    epoch: int,
) -> float:
    model.train()
    total_loss = 0.0
    total_examples = 0
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        loss = criterion(outputs["logits"], batch["label"])
        loss.backward()
        optimizer.step()
        batch_size = batch["label"].shape[0]
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size
    return total_loss / max(total_examples, 1)


@torch.no_grad()
def predict(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    criterion: Optional[nn.Module] = None,
    desc: Optional[str] = None,
) -> Tuple[Optional[float], pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    rows = []
    for batch in loader:  # ← plain loop, no inner tqdm
        batch = move_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        logits = outputs["logits"]
        probs = torch.sigmoid(logits)
        if criterion is not None:
            loss = criterion(logits, batch["label"])
            batch_size = batch["label"].shape[0]
            total_loss += float(loss.item()) * batch_size
            total_examples += batch_size
        for idx in range(len(batch["sequence_id"])):
            rows.append(
                {
                    "sequence_id": batch["sequence_id"][idx],
                    "sequence": batch["sequence"][idx],
                    "label": int(batch["label"][idx].item()),
                    "logit": float(logits[idx].item()),
                    "pred_prob": float(probs[idx].item()),
                    "pred_label": int(probs[idx].item() >= THRESHOLD),
                }
            )
    loss_value = None if criterion is None else total_loss / max(total_examples, 1)
    return loss_value, pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label"].to_numpy()
    y_prob = pred_df["pred_prob"].to_numpy()
    y_pred = pred_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "auprc": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else math.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    return metrics


history = []
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

_epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch", file=sys.stdout, dynamic_ncols=True)
for epoch in _epoch_bar:
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch)
    val_loss, val_pred_df = predict(
        model, val_loader, DEVICE, criterion,
    )
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "esm_model_name": ESM_MODEL_NAME,
                "architecture_hyperparameters": {
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                },
                "training_history": history,
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    # ← print each epoch result as a plain line so you always see progress
    print(
        f"Epoch {epoch:>3}/{EPOCHS} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"best={best_epoch}"
    )
    _epoch_bar.set_postfix(
        train_loss=f"{train_loss:.5f}",
        val_loss=f"{val_loss:.5f}",
        best=best_epoch,
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print(f"Best validation loss: {best_val_loss:.5f} at epoch {best_epoch}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")

Training:   0%|          | 0/15 [00:00<?, ?epoch/s]

Epoch   1/15 | train_loss=0.68189 | val_loss=0.66785 | best=1
Epoch   2/15 | train_loss=0.64748 | val_loss=0.62333 | best=2
Epoch   3/15 | train_loss=0.58966 | val_loss=0.55831 | best=3
Epoch   4/15 | train_loss=0.51977 | val_loss=0.49434 | best=4
Epoch   5/15 | train_loss=0.45680 | val_loss=0.43856 | best=5
Epoch   6/15 | train_loss=0.40182 | val_loss=0.39861 | best=6
Epoch   7/15 | train_loss=0.36297 | val_loss=0.37106 | best=7
Epoch   8/15 | train_loss=0.33463 | val_loss=0.35115 | best=8
Epoch   9/15 | train_loss=0.31371 | val_loss=0.33641 | best=9
Epoch  10/15 | train_loss=0.29659 | val_loss=0.32429 | best=10
Epoch  11/15 | train_loss=0.28449 | val_loss=0.31564 | best=11
Epoch  12/15 | train_loss=0.27524 | val_loss=0.30889 | best=12
Epoch  13/15 | train_loss=0.26559 | val_loss=0.30379 | best=13
Epoch  14/15 | train_loss=0.25932 | val_loss=0.29990 | best=14
Epoch  15/15 | train_loss=0.25178 | val_loss=0.29634 | best=15
Best validation loss: 0.29634 at epoch 15
Checkpoint saved to: /

In [10]:
model, checkpoint = load_finetune_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
)

train_history_df = pd.DataFrame(checkpoint["training_history"])

val_loss, val_predictions_df = predict(model, val_loader, DEVICE, criterion)
test_loss, test_predictions_df = predict(model, test_loader, DEVICE, criterion)

test_metrics = compute_metrics(test_predictions_df)
test_metrics["test_loss"] = float(test_loss)
test_metrics["best_epoch"] = int(best_epoch)
test_metrics["n_test_sequences"] = int(len(test_predictions_df))

metrics_payload = {
    "esm_model_name": ESM_MODEL_NAME,
    "architecture_hyperparameters": {"hidden_dim": HIDDEN_DIM, "dropout": DROPOUT},
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    },
    "validation_loss": float(val_loss),
    "test_metrics": test_metrics,
}

with METRICS_PATH.open("w") as handle:
    json.dump(metrics_payload, handle, indent=2)


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
print('Runtime outputs:')
print(f'  Checkpoint : {CHECKPOINT_PATH}')
print(f'  Metrics    : {METRICS_PATH}')
print(f'  (Will be copied to Drive in the final cell.)')


Runtime outputs:
  Checkpoint : /content/XAllergen2.0/models/finetuned_esm2_8e-6.pt
  Metrics    : /content/XAllergen2.0/results/finetuned_esm2_8e-6_metrics.json
  (Will be copied to Drive in the final cell.)


In [12]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_history_df["epoch"], train_history_df["train_loss"], label="Train loss")
axes[0].plot(train_history_df["epoch"], train_history_df["val_loss"], label="Validation loss")
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

cm = confusion_matrix(test_predictions_df["label"], test_predictions_df["pred_label"], labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("Held-out test confusion matrix")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
axes[1].set_xticklabels(["Negative", "Positive"])
axes[1].set_yticklabels(["Negative", "Positive"], rotation=0)

plt.tight_layout()
plt.show()

display(Markdown("## Test metrics"))
display(pd.DataFrame([{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}]).T.rename(columns={0: 'value'}))
print(f"Saved metrics JSON to: {METRICS_PATH}")

## Test metrics

,value
threshold,0.500000
accuracy,0.901163
auroc,0.955588
auprc,0.954347
f1,0.899705
precision,0.913174
recall,0.886628
specificity,0.915698
mcc,0.802665
test_loss,0.268009


Saved metrics JSON to: /content/XAllergen2.0/results/finetuned_esm2_8e-6_metrics.json


In [13]:
def select_attribution_examples(pred_df: pd.DataFrame) -> pd.DataFrame:
    selected_frames = []
    selected_ids = set()

    high_conf_tp = pred_df.loc[
        (pred_df["label"] == 1)
        & (pred_df["pred_label"] == 1)
        & (pred_df["pred_prob"] > 0.9)
    ]
    if len(high_conf_tp) > 0:
        tp_sample = high_conf_tp.sample(n=min(3, len(high_conf_tp)), random_state=RANDOM_STATE)
        selected_frames.append(tp_sample)
        selected_ids.update(tp_sample["sequence_id"])

    errors = pred_df.loc[pred_df["pred_label"] != pred_df["label"]]
    if len(errors) > 0:
        error_sample = errors.sample(n=min(2, len(errors)), random_state=RANDOM_STATE)
        selected_frames.append(error_sample)
        selected_ids.update(error_sample["sequence_id"])

    selected_df = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame(columns=pred_df.columns)

    if len(selected_df) < 5:
        remainder = pred_df.loc[~pred_df["sequence_id"].isin(selected_ids)]
        fill_n = min(5 - len(selected_df), len(remainder))
        if fill_n > 0:
            filler = remainder.sample(n=fill_n, random_state=RANDOM_STATE)
            selected_df = pd.concat([selected_df, filler], ignore_index=True)

    return selected_df.head(5).copy()


def compute_attention_and_ig(sequence: str) -> tuple[np.ndarray, np.ndarray]:
    attention_scores = normalize_scores(
        compute_attention_weights(model, tokenizer, sequence, DEVICE)
    )
    ig_scores = compute_integrated_gradients(
        model,
        tokenizer,
        sequence,
        DEVICE,
        steps=IG_STEPS,
        normalize=True,
    )
    return attention_scores, ig_scores


In [14]:
import shutil

# Copy checkpoint and metrics JSON to Google Drive
shutil.copy(CHECKPOINT_PATH, DRIVE_MODELS / CHECKPOINT_PATH.name)
shutil.copy(METRICS_PATH,    DRIVE_RESULTS / METRICS_PATH.name)

print('Saved to Google Drive:')
print(f'  {DRIVE_MODELS / CHECKPOINT_PATH.name}')
print(f'  {DRIVE_RESULTS / METRICS_PATH.name}')


Saved to Google Drive:
  /content/drive/MyDrive/XAllergen2.0/models/finetuned_esm2_8e-6.pt
  /content/drive/MyDrive/XAllergen2.0/results/finetuned_esm2_8e-6_metrics.json


## Wrap-up

The fine-tuned ESM-2 checkpoint and held-out test metrics have been saved to `/content/XAllergen2.0/` on the Colab runtime **and** copied to your Google Drive under `MyDrive/XAllergen2.0/`. Review the evaluation plots and test metrics above as the fine-tuning reference point.

The next step is dataset-scale attribution evaluation against IEDB epitope ground truth in `03_attribution_evaluation.ipynb`.
